In [ ]:
# ============================================================
# CELL 1 — MOUNT GOOGLE DRIVE + ENVIRONMENT CHECK
# ============================================================

import os
import sys
import time
import shutil
import subprocess
from pathlib import Path

print("=" * 70)
print("NOTEBOOK 8 — RT-DETR-X OBJECT DETECTION")
print("=" * 70)

# ------------------------------------------------------------
# MOUNT GOOGLE DRIVE
# ------------------------------------------------------------

from google.colab import drive

DRIVE_MOUNT = "/content/drive"

# Avoid mount error if already mounted
if Path("/content/drive/MyDrive").exists():
    print("\n✓ Google Drive already mounted")

else:
    if Path(DRIVE_MOUNT).exists():
        try:
            if not os.listdir(DRIVE_MOUNT):
                pass
            else:
                shutil.rmtree(DRIVE_MOUNT)
        except:
            pass

    drive.mount(DRIVE_MOUNT)

print("\nGoogle Drive:")
print(Path("/content/drive/MyDrive"))

# ------------------------------------------------------------
# GPU CHECK
# ------------------------------------------------------------

import torch

print("\n" + "=" * 70)
print("GPU CHECK")
print("=" * 70)

print("\nPyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "\n❌ GPU NOT AVAILABLE.\n"
    "STOP HERE.\n"
    "Go to Runtime → Change runtime type → T4 GPU.\n"
    "Then reconnect and run this notebook again."
)

print("GPU:", torch.cuda.get_device_name(0))

print("\n" + "=" * 70)
print("✓ ENVIRONMENT READY")
print("=" * 70)

NOTEBOOK 8 — RT-DETR-X OBJECT DETECTION
Mounted at /content/drive

Google Drive:
/content/drive/MyDrive

GPU CHECK

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

✓ ENVIRONMENT READY


In [ ]:
# ============================================================
# CELL 2 — INSTALL / VERIFY ULTRALYTICS
# ============================================================

print("=" * 70)
print("INSTALLING / VERIFYING ULTRALYTICS")
print("=" * 70)

!pip install -q -U ultralytics

import ultralytics

print("\nUltralytics version:", ultralytics.__version__)

from ultralytics import RTDETR

print("\n✓ ULTRALYTICS READY")

INSTALLING / VERIFYING ULTRALYTICS
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.

Ultralytics version: 8.4.142

✓ ULTRALYTICS READY


In [ ]:
# ============================================================
# CELL 3 — PROJECT PATHS + DATASET ZIP DISCOVERY
# ============================================================

from pathlib import Path

print("=" * 70)
print("RT-DETR PROJECT PATH CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Image_captioning"
)

assert PROJECT_ROOT.exists(), (
    f"Project not found:\n{PROJECT_ROOT}"
)

# ------------------------------------------------------------
# REQUIRED CAPTIONS FILE
# ------------------------------------------------------------

CAPTIONS_FILE = (
    PROJECT_ROOT /
    "Dataset/Processed/Captions/captions_processed.csv"
)

print("\nPROJECT ROOT:")
print(PROJECT_ROOT)

print("\nCAPTIONS FILE:")
print(CAPTIONS_FILE)

print("Exists:", CAPTIONS_FILE.exists())

assert CAPTIONS_FILE.exists(), (
    "Processed captions file not found."
)

# ------------------------------------------------------------
# FIND ZIP FILES AUTOMATICALLY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SEARCHING FOR DATASET ZIP FILES")
print("=" * 70)

zip_files = list(
    PROJECT_ROOT.rglob("*.zip")
)

print("\nZIP files found:", len(zip_files))

for file in zip_files:
    print(file)

assert len(zip_files) > 0, (
    "No ZIP files found anywhere inside the project."
)

# ------------------------------------------------------------
# IDENTIFY FLICKR8K ZIP
# ------------------------------------------------------------

flickr8k_candidates = [
    p for p in zip_files
    if "flickr8k" in p.name.lower()
]

# ------------------------------------------------------------
# IDENTIFY FLICKR30K ZIP
# ------------------------------------------------------------

flickr30k_candidates = [
    p for p in zip_files
    if "flickr30k" in p.name.lower()
]

print("\nFlickr8K ZIP candidates:")
for p in flickr8k_candidates:
    print(p)

print("\nFlickr30K ZIP candidates:")
for p in flickr30k_candidates:
    print(p)

assert len(flickr8k_candidates) >= 1, (
    "Could not automatically find Flickr8K ZIP."
)

assert len(flickr30k_candidates) >= 1, (
    "Could not automatically find Flickr30K ZIP."
)

FLICKR8K_ZIP = flickr8k_candidates[0]
FLICKR30K_ZIP = flickr30k_candidates[0]

print("\n" + "=" * 70)
print("DATASET ZIP FILES SELECTED")
print("=" * 70)

print("\nFlickr8K ZIP:")
print(FLICKR8K_ZIP)

print("\nFlickr30K ZIP:")
print(FLICKR30K_ZIP)

print("\n✓ ZIP FILES VERIFIED")

RT-DETR PROJECT PATH CONFIGURATION

PROJECT ROOT:
/content/drive/MyDrive/Image_captioning

CAPTIONS FILE:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Captions/captions_processed.csv
Exists: True

SEARCHING FOR DATASET ZIP FILES

ZIP files found: 2
/content/drive/MyDrive/Image_captioning/Dataset/ZIP/flickr8k.zip
/content/drive/MyDrive/Image_captioning/Dataset/ZIP/flickr30k.zip

Flickr8K ZIP candidates:
/content/drive/MyDrive/Image_captioning/Dataset/ZIP/flickr8k.zip

Flickr30K ZIP candidates:
/content/drive/MyDrive/Image_captioning/Dataset/ZIP/flickr30k.zip

DATASET ZIP FILES SELECTED

Flickr8K ZIP:
/content/drive/MyDrive/Image_captioning/Dataset/ZIP/flickr8k.zip

Flickr30K ZIP:
/content/drive/MyDrive/Image_captioning/Dataset/ZIP/flickr30k.zip

✓ ZIP FILES VERIFIED


In [ ]:
# ============================================================
# CELL 4 — EXTRACT DATASETS TO LOCAL COLAB STORAGE
# ============================================================

print("=" * 70)
print("LOCAL DATASET EXTRACTION")
print("=" * 70)

# ------------------------------------------------------------
# LOCAL ROOT
# ------------------------------------------------------------

LOCAL_ROOT = Path(
    "/content/Image_captioning_Local"
)

LOCAL_FLICKR8K_ROOT = (
    LOCAL_ROOT / "Flickr8K"
)

LOCAL_FLICKR30K_ROOT = (
    LOCAL_ROOT / "Flickr30K"
)

# ------------------------------------------------------------
# CREATE ROOT
# ------------------------------------------------------------

LOCAL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# DISK SPACE
# ------------------------------------------------------------

print("\nLOCAL DISK SPACE BEFORE EXTRACTION:\n")

subprocess.run(
    ["df", "-h", "/content"]
)

# ------------------------------------------------------------
# EXTRACT FUNCTION
# ------------------------------------------------------------

def extract_zip_to_local(
    zip_path,
    destination,
    dataset_name
):

    print("\n" + "=" * 70)
    print(f"EXTRACTING {dataset_name}")
    print("=" * 70)

    print("\nZIP:")
    print(zip_path)

    print("\nDestination:")
    print(destination)

    # Remove incomplete extraction
    if destination.exists():

        existing_images = list(
            destination.rglob("*.jpg")
        )

        print(
            "\nExisting local JPG files:",
            len(existing_images)
        )

    else:

        destination.mkdir(
            parents=True,
            exist_ok=True
        )

        existing_images = []

    start = time.time()

    result = subprocess.run(
        [
            "unzip",
            "-q",
            str(zip_path),
            "-d",
            str(destination)
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    elapsed = (
        time.time() - start
    )

    if result.returncode != 0:

        print("\nUNZIP ERROR:")
        print(result.stderr)

        raise RuntimeError(
            f"Extraction failed for {dataset_name}"
        )

    print(
        f"\nExtraction time: "
        f"{elapsed / 60:.2f} minutes"
    )

    images = list(
        destination.rglob("*.jpg")
    )

    print(
        "\nJPG files found:",
        len(images)
    )

    if len(images) > 0:

        print(
            "\nSample image:"
        )

        print(images[0])

    return images


# ------------------------------------------------------------
# FLICKR8K
# ------------------------------------------------------------

flickr8k_images = extract_zip_to_local(
    FLICKR8K_ZIP,
    LOCAL_FLICKR8K_ROOT,
    "Flickr8K"
)

# ------------------------------------------------------------
# FLICKR30K
# ------------------------------------------------------------

flickr30k_images = extract_zip_to_local(
    FLICKR30K_ZIP,
    LOCAL_FLICKR30K_ROOT,
    "Flickr30K"
)

# ------------------------------------------------------------
# FINAL LOCAL CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOCAL EXTRACTION COMPLETE")
print("=" * 70)

print(
    "\nFlickr8K JPG files:",
    len(flickr8k_images)
)

print(
    "\nFlickr30K JPG files:",
    len(flickr30k_images)
)

assert len(flickr8k_images) >= 8091, (
    f"Expected at least 8,091 Flickr8K images. "
    f"Found {len(flickr8k_images)}"
)

# Flickr30K ZIP may contain duplicate nested folders,
# so we verify at least expected image count.
assert len(flickr30k_images) >= 31783, (
    f"Expected at least 31,783 Flickr30K images. "
    f"Found {len(flickr30k_images)}"
)

print("\n✓ BOTH DATASETS READY LOCALLY")

LOCAL DATASET EXTRACTION

LOCAL DISK SPACE BEFORE EXTRACTION:


EXTRACTING Flickr8K

ZIP:
/content/drive/MyDrive/Image_captioning/Dataset/ZIP/flickr8k.zip

Destination:
/content/Image_captioning_Local/Flickr8K

Extraction time: 0.48 minutes

JPG files found: 8091

Sample image:
/content/Image_captioning_Local/Flickr8K/Images/2661437618_ca7a15f3cb.jpg

EXTRACTING Flickr30K

ZIP:
/content/drive/MyDrive/Image_captioning/Dataset/ZIP/flickr30k.zip

Destination:
/content/Image_captioning_Local/Flickr30K

Extraction time: 3.78 minutes

JPG files found: 63566

Sample image:
/content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/1177949654.jpg

LOCAL EXTRACTION COMPLETE

Flickr8K JPG files: 8091

Flickr30K JPG files: 63566

✓ BOTH DATASETS READY LOCALLY


In [ ]:
# ============================================================
# CELL 5 — BUILD RT-DETR IMAGE MANIFEST
# ============================================================

import pandas as pd

print("=" * 70)
print("BUILDING IMAGE MANIFEST")
print("=" * 70)

# ------------------------------------------------------------
# LOAD PROCESSED CAPTIONS
# ------------------------------------------------------------

captions_df = pd.read_csv(
    CAPTIONS_FILE
)

print("\nCaption records:")
print(len(captions_df))

print("\nColumns:")
print(
    captions_df.columns.tolist()
)

# ------------------------------------------------------------
# GET UNIQUE IMAGES
# ------------------------------------------------------------

image_manifest = (
    captions_df[
        [
            "image_id",
            "dataset",
            "image_name"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nTotal unique images:")
print(len(image_manifest))

print("\nDataset distribution:")

print(
    image_manifest[
        "dataset"
    ].value_counts()
)

assert len(image_manifest) == 39874, (
    f"Expected 39,874 images, "
    f"found {len(image_manifest)}"
)

# ------------------------------------------------------------
# BUILD FLICKR8K MAP
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BUILDING FLICKR8K PATH MAP")
print("=" * 70)

flickr8k_files = list(
    LOCAL_FLICKR8K_ROOT.rglob("*.jpg")
)

flickr8k_map = {}

for path in flickr8k_files:

    if path.name not in flickr8k_map:

        flickr8k_map[
            path.name
        ] = str(path)

print(
    "\nUnique Flickr8K mapped files:",
    len(flickr8k_map)
)

# ------------------------------------------------------------
# BUILD FLICKR30K MAP
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BUILDING FLICKR30K PATH MAP")
print("=" * 70)

flickr30k_files = list(
    LOCAL_FLICKR30K_ROOT.rglob("*.jpg")
)

flickr30k_map = {}

for path in flickr30k_files:

    # Some ZIP structures contain duplicates.
    # Keep the first path for each filename.
    if path.name not in flickr30k_map:

        flickr30k_map[
            path.name
        ] = str(path)

print(
    "\nUnique Flickr30K mapped files:",
    len(flickr30k_map)
)

# ------------------------------------------------------------
# ASSIGN LOCAL PATH
# ------------------------------------------------------------

def get_local_path(row):

    if row["dataset"] == "Flickr8K":

        return flickr8k_map.get(
            row["image_name"]
        )

    elif row["dataset"] == "Flickr30K":

        return flickr30k_map.get(
            row["image_name"]
        )

    return None


image_manifest[
    "local_image_path"
] = image_manifest.apply(
    get_local_path,
    axis=1
)

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

missing_images = (
    image_manifest[
        image_manifest[
            "local_image_path"
        ].isna()
    ]
)

print("\n" + "=" * 70)
print("LOCAL IMAGE VERIFICATION")
print("=" * 70)

print(
    "\nTotal images:",
    len(image_manifest)
)

print(
    "\nImages missing locally:",
    len(missing_images)
)

if len(missing_images) > 0:

    print(
        missing_images.head(20)
    )

assert len(missing_images) == 0, (
    "Some manifest images were not found locally."
)

print("\n✓ ALL 39,874 IMAGES MAPPED LOCALLY")

BUILDING IMAGE MANIFEST

Caption records:
199369

Columns:
['image_id', 'dataset', 'image_name', 'caption', 'processed_caption', 'model_caption', 'caption_length']

Total unique images:
39874

Dataset distribution:
dataset
Flickr30K    31783
Flickr8K      8091
Name: count, dtype: int64

BUILDING FLICKR8K PATH MAP

Unique Flickr8K mapped files: 8091

BUILDING FLICKR30K PATH MAP

Unique Flickr30K mapped files: 31783

LOCAL IMAGE VERIFICATION

Total images: 39874

Images missing locally: 0

✓ ALL 39,874 IMAGES MAPPED LOCALLY


In [ ]:
# ============================================================
# CELL 6 — LOAD RT-DETR-X + SINGLE IMAGE TEST
# ============================================================

print("=" * 70)
print("LOADING RT-DETR-X")
print("=" * 70)

MODEL_NAME = "rtdetr-x.pt"

# Higher accuracy RT-DETR variant
rtdetr_model = RTDETR(
    MODEL_NAME
)

print(
    "\nModel loaded:"
)

print(
    MODEL_NAME
)

# ------------------------------------------------------------
# TEST IMAGE
# ------------------------------------------------------------

test_row = (
    image_manifest
    .iloc[0]
)

test_path = (
    test_row[
        "local_image_path"
    ]
)

print("\n" + "=" * 70)
print("RT-DETR SINGLE IMAGE TEST")
print("=" * 70)

print(
    "\nImage ID:"
)

print(
    test_row["image_id"]
)

print(
    "\nImage path:"
)

print(
    test_path
)

start = time.time()

test_results = rtdetr_model.predict(
    source=test_path,
    conf=0.25,
    imgsz=640,
    device=0,
    verbose=False
)

elapsed = (
    time.time()
    - start
)

result = (
    test_results[0]
)

boxes = result.boxes

print(
    "\nInference time:"
)

print(
    f"{elapsed:.2f} seconds"
)

print(
    "\nDetections found:"
)

print(
    len(boxes)
)

if len(boxes) > 0:

    print(
        "\nDetections:"
    )

    names = result.names

    for i in range(len(boxes)):

        cls_id = int(
            boxes.cls[i].item()
        )

        confidence = float(
            boxes.conf[i].item()
        )

        print(
            f"{i}: "
            f"{names[cls_id]} | "
            f"{confidence:.4f}"
        )

print("\n✓ SINGLE IMAGE TEST COMPLETE")

LOADING RT-DETR-X

Model loaded:
rtdetr-x.pt

RT-DETR SINGLE IMAGE TEST

Image ID:
Flickr8K/1000268201_693b08cb0e.jpg

Image path:
/content/Image_captioning_Local/Flickr8K/Images/1000268201_693b08cb0e.jpg

Inference time:
9.86 seconds

Detections found:
7

Detections:
0: person | 0.9520
1: bench | 0.4993
2: bench | 0.4631
3: bench | 0.3171
4: potted plant | 0.3106
5: bench | 0.2861
6: bottle | 0.2638

✓ SINGLE IMAGE TEST COMPLETE


In [ ]:
# ============================================================
# CELL 7 — RT-DETR-X LOCAL GPU SPEED TEST
# ============================================================

print("=" * 70)
print("RT-DETR-X SPEED TEST")
print("=" * 70)

TEST_BATCH_SIZE = 16

test_batch = (
    image_manifest
    .head(TEST_BATCH_SIZE)
    .copy()
)

test_paths = (
    test_batch[
        "local_image_path"
    ]
    .tolist()
)

print(
    "\nImages in test batch:",
    len(test_paths)
)

# ------------------------------------------------------------
# GPU MEMORY RESET
# ------------------------------------------------------------

torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()

# ------------------------------------------------------------
# RUN TEST
# ------------------------------------------------------------

start = time.time()

results = rtdetr_model.predict(
    source=test_paths,
    conf=0.25,
    imgsz=640,
    device=0,
    verbose=False
)

elapsed = (
    time.time()
    - start
)

total_detections = sum(
    len(result.boxes)
    for result in results
)

speed = (
    len(test_paths)
    / elapsed
)

memory_gb = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 3)
)

print("\n" + "=" * 70)
print("RT-DETR-X SPEED RESULTS")
print("=" * 70)

print(
    "\nImages:",
    len(test_paths)
)

print(
    "\nDetection records:",
    total_detections
)

print(
    "\nTime:"
)

print(
    f"{elapsed:.2f} seconds"
)

print(
    "\nSpeed:"
)

print(
    f"{speed:.2f} images/second"
)

print(
    "\nPeak GPU memory:"
)

print(
    f"{memory_gb:.2f} GB"
)

print("\n✓ SPEED TEST COMPLETE")

RT-DETR-X SPEED TEST

Images in test batch: 16

RT-DETR-X SPEED RESULTS

Images: 16

Detection records: 65

Time:
1.07 seconds

Speed:
14.89 images/second

Peak GPU memory:
2.29 GB

✓ SPEED TEST COMPLETE


In [ ]:
# ============================================================
# CELL 8 — RT-DETR OUTPUT CONFIGURATION
# ============================================================

print("=" * 70)
print("RT-DETR OUTPUT CONFIGURATION")
print("=" * 70)

RTDETR_OUTPUT_ROOT = (
    PROJECT_ROOT /
    "Outputs/RTDETR_Detection"
)

CHECKPOINT_DIR = (
    RTDETR_OUTPUT_ROOT /
    "Checkpoints"
)

FINAL_DIR = (
    RTDETR_OUTPUT_ROOT /
    "Final"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_OUTPUT_FILE = (
    FINAL_DIR /
    "rtdetr_x_detections_full.csv"
)

print(
    "\nOutput root:"
)

print(
    RTDETR_OUTPUT_ROOT
)

print(
    "\nCheckpoint directory:"
)

print(
    CHECKPOINT_DIR
)

print(
    "\nFinal output:"
)

print(
    FINAL_OUTPUT_FILE
)

print("\n✓ OUTPUT DIRECTORIES READY")

RT-DETR OUTPUT CONFIGURATION

Output root:
/content/drive/MyDrive/Image_captioning/Outputs/RTDETR_Detection

Checkpoint directory:
/content/drive/MyDrive/Image_captioning/Outputs/RTDETR_Detection/Checkpoints

Final output:
/content/drive/MyDrive/Image_captioning/Outputs/RTDETR_Detection/Final/rtdetr_x_detections_full.csv

✓ OUTPUT DIRECTORIES READY


In [ ]:
# ============================================================
# CELL 9 — RESTORE RT-DETR CHECKPOINT STATE
# ============================================================

import re

print("=" * 70)
print("RESTORING RT-DETR CHECKPOINT STATE")
print("=" * 70)

checkpoint_files = sorted(
    CHECKPOINT_DIR.glob(
        "checkpoint_*.csv"
    ),
    key=lambda p: int(
        re.search(
            r"(\d+)",
            p.stem
        ).group()
    )
)

print(
    "\nCheckpoint files found:",
    len(checkpoint_files)
)

for file in checkpoint_files:
    print(
        file.name
    )

# ------------------------------------------------------------
# LOAD COMPLETED IDS
# ------------------------------------------------------------

processed_ids = set()

for file in checkpoint_files:

    checkpoint_df = pd.read_csv(
        file
    )

    if "image_id" not in checkpoint_df.columns:

        raise ValueError(
            f"image_id column missing in {file.name}"
        )

    processed_ids.update(
        checkpoint_df[
            "image_id"
        ].dropna()
    )

print(
    "\nCompleted unique images:",
    len(processed_ids)
)

# ------------------------------------------------------------
# REMAINING IMAGES
# ------------------------------------------------------------

remaining_manifest = (
    image_manifest[
        ~image_manifest[
            "image_id"
        ].isin(
            processed_ids
        )
    ]
    .reset_index(drop=True)
)

print(
    "\nTotal dataset images:",
    len(image_manifest)
)

print(
    "\nAlready completed:",
    len(processed_ids)
)

print(
    "\nRemaining:",
    len(remaining_manifest)
)

# ------------------------------------------------------------
# NEXT CHECKPOINT NUMBER
# ------------------------------------------------------------

if len(checkpoint_files) == 0:

    next_checkpoint_number = 1

else:

    last_number = max(
        int(
            re.search(
                r"(\d+)",
                file.stem
            ).group()
        )
        for file in checkpoint_files
    )

    next_checkpoint_number = (
        last_number + 1
    )

print(
    "\nNext checkpoint number:",
    next_checkpoint_number
)

print("\n✓ RESUME STATE READY")

RESTORING RT-DETR CHECKPOINT STATE

Checkpoint files found: 0

Completed unique images: 0

Total dataset images: 39874

Already completed: 0

Remaining: 39874

Next checkpoint number: 1

✓ RESUME STATE READY


In [ ]:
# ============================================================
# CELL 10 — FULL RT-DETR-X DATASET PROCESSING
# ============================================================

from tqdm.auto import tqdm
import numpy as np
import gc

print("=" * 70)
print("RT-DETR-X FULL DATASET PROCESSING")
print("=" * 70)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

INFERENCE_BATCH_SIZE = 16

CHECKPOINT_INTERVAL = 512

CONFIDENCE_THRESHOLD = 0.25

IMAGE_SIZE = 640

TOTAL_REMAINING = (
    len(remaining_manifest)
)

print(
    "\nModel:",
    MODEL_NAME
)

print(
    "\nGPU:",
    torch.cuda.get_device_name(0)
)

print(
    "\nInference batch size:",
    INFERENCE_BATCH_SIZE
)

print(
    "\nCheckpoint interval:",
    CHECKPOINT_INTERVAL
)

print(
    "\nConfidence threshold:",
    CONFIDENCE_THRESHOLD
)

print(
    "\nImages remaining:",
    TOTAL_REMAINING
)

# ------------------------------------------------------------
# OUTPUT COLUMNS
# ------------------------------------------------------------

OUTPUT_COLUMNS = [
    "image_id",
    "image_path",
    "object_name",
    "confidence",
    "xmin",
    "ymin",
    "xmax",
    "ymax",
    "objects_detected"
]

# ------------------------------------------------------------
# PROCESS FUNCTION
# ------------------------------------------------------------

def process_batch(
    batch_df
):

    paths = (
        batch_df[
            "local_image_path"
        ]
        .tolist()
    )

    results = rtdetr_model.predict(
        source=paths,
        conf=CONFIDENCE_THRESHOLD,
        imgsz=IMAGE_SIZE,
        device=0,
        verbose=False
    )

    records = []

    for row_dict, result in zip(
        batch_df.to_dict("records"),
        results
    ):

        image_id = (
            row_dict["image_id"]
        )

        image_path = (
            row_dict[
                "local_image_path"
            ]
        )

        boxes = result.boxes

        # --------------------------------------------
        # ZERO DETECTION IMAGE
        # --------------------------------------------

        if boxes is None or len(boxes) == 0:

            records.append({

                "image_id": image_id,

                "image_path": image_path,

                "object_name": None,

                "confidence": np.nan,

                "xmin": np.nan,

                "ymin": np.nan,

                "xmax": np.nan,

                "ymax": np.nan,

                "objects_detected": 0
            })

            continue

        names = (
            result.names
        )

        object_count = (
            len(boxes)
        )

        # --------------------------------------------
        # DETECTIONS
        # --------------------------------------------

        for i in range(
            object_count
        ):

            cls_id = int(
                boxes.cls[i].item()
            )

            confidence = float(
                boxes.conf[i].item()
            )

            coords = (
                boxes.xyxy[i]
                .cpu()
                .numpy()
            )

            records.append({

                "image_id": image_id,

                "image_path": image_path,

                "object_name":
                    names[cls_id],

                "confidence":
                    confidence,

                "xmin":
                    float(coords[0]),

                "ymin":
                    float(coords[1]),

                "xmax":
                    float(coords[2]),

                "ymax":
                    float(coords[3]),

                "objects_detected":
                    object_count
            })

    return records


# ------------------------------------------------------------
# MAIN PROCESSING
# ------------------------------------------------------------

all_checkpoint_records = []

images_since_checkpoint = 0

checkpoint_number = (
    next_checkpoint_number
)

start_time = time.time()

pbar = tqdm(
    total=TOTAL_REMAINING,
    desc="RT-DETR-X Detection"
)

try:

    for start_idx in range(
        0,
        TOTAL_REMAINING,
        INFERENCE_BATCH_SIZE
    ):

        end_idx = min(
            start_idx +
            INFERENCE_BATCH_SIZE,
            TOTAL_REMAINING
        )

        batch_df = (
            remaining_manifest
            .iloc[
                start_idx:end_idx
            ]
            .copy()
        )

        # ------------------------------------------------
        # PROCESS BATCH
        # ------------------------------------------------

        batch_records = (
            process_batch(
                batch_df
            )
        )

        all_checkpoint_records.extend(
            batch_records
        )

        images_processed = (
            len(batch_df)
        )

        images_since_checkpoint += (
            images_processed
        )

        pbar.update(
            images_processed
        )

        # ------------------------------------------------
        # SAVE CHECKPOINT
        # ------------------------------------------------

        if (
            images_since_checkpoint
            >= CHECKPOINT_INTERVAL
        ):

            checkpoint_df = (
                pd.DataFrame(
                    all_checkpoint_records,
                    columns=OUTPUT_COLUMNS
                )
            )

            checkpoint_path = (
                CHECKPOINT_DIR /
                f"checkpoint_{checkpoint_number:05d}.csv"
            )

            checkpoint_df.to_csv(
                checkpoint_path,
                index=False
            )

            unique_checkpoint_images = (
                checkpoint_df[
                    "image_id"
                ]
                .nunique()
            )

            print(
                f"\n✓ Checkpoint "
                f"{checkpoint_number} saved"
            )

            print(
                "Images in checkpoint:",
                unique_checkpoint_images
            )

            print(
                "Rows:",
                len(checkpoint_df)
            )

            # Reset
            all_checkpoint_records = []

            images_since_checkpoint = 0

            checkpoint_number += 1

            gc.collect()

            torch.cuda.empty_cache()


except KeyboardInterrupt:

    print(
        "\n\n⚠ PROCESS INTERRUPTED"
    )

    print(
        "Saving current progress..."
    )

    if len(all_checkpoint_records) > 0:

        checkpoint_df = (
            pd.DataFrame(
                all_checkpoint_records,
                columns=OUTPUT_COLUMNS
            )
        )

        checkpoint_path = (
            CHECKPOINT_DIR /
            f"checkpoint_{checkpoint_number:05d}.csv"
        )

        checkpoint_df.to_csv(
            checkpoint_path,
            index=False
        )

        print(
            "\n✓ INTERRUPTED PROGRESS SAVED:"
        )

        print(
            checkpoint_path
        )

    raise


# ------------------------------------------------------------
# SAVE FINAL PARTIAL CHECKPOINT
# ------------------------------------------------------------

if len(all_checkpoint_records) > 0:

    checkpoint_df = (
        pd.DataFrame(
            all_checkpoint_records,
            columns=OUTPUT_COLUMNS
        )
    )

    checkpoint_path = (
        CHECKPOINT_DIR /
        f"checkpoint_{checkpoint_number:05d}.csv"
    )

    checkpoint_df.to_csv(
        checkpoint_path,
        index=False
    )

    print(
        f"\n✓ Final checkpoint "
        f"{checkpoint_number} saved"
    )

    print(
        "Images in checkpoint:",
        checkpoint_df[
            "image_id"
        ].nunique()
    )


pbar.close()

elapsed = (
    time.time()
    - start_time
)

print("\n" + "=" * 70)
print("RT-DETR-X PROCESSING COMPLETE")
print("=" * 70)

print(
    "\nProcessing time:"
)

print(
    f"{elapsed / 60:.2f} minutes"
)

print(
    "\nAverage speed:"
)

if TOTAL_REMAINING > 0:

    print(
        f"{TOTAL_REMAINING / elapsed:.2f} "
        f"images/second"
    )

print(
    "\nCheckpoints saved:"
)

print(
    len(
        list(
            CHECKPOINT_DIR.glob(
                "checkpoint_*.csv"
            )
        )
    )
)

RT-DETR-X FULL DATASET PROCESSING

Model: rtdetr-x.pt

GPU: Tesla T4

Inference batch size: 16

Checkpoint interval: 512

Confidence threshold: 0.25

Images remaining: 39874


RT-DETR-X Detection:   0%|          | 0/39874 [00:00<?, ?it/s]


✓ Checkpoint 1 saved
Images in checkpoint: 512
Rows: 4059

✓ Checkpoint 2 saved
Images in checkpoint: 512
Rows: 3634

✓ Checkpoint 3 saved
Images in checkpoint: 512
Rows: 3686

✓ Checkpoint 4 saved
Images in checkpoint: 512
Rows: 4496

✓ Checkpoint 5 saved
Images in checkpoint: 512
Rows: 3776

✓ Checkpoint 6 saved
Images in checkpoint: 512
Rows: 4115

✓ Checkpoint 7 saved
Images in checkpoint: 512
Rows: 4286

✓ Checkpoint 8 saved
Images in checkpoint: 512
Rows: 5189

✓ Checkpoint 9 saved
Images in checkpoint: 512
Rows: 4959

✓ Checkpoint 10 saved
Images in checkpoint: 512
Rows: 5109

✓ Checkpoint 11 saved
Images in checkpoint: 512
Rows: 4753

✓ Checkpoint 12 saved
Images in checkpoint: 512
Rows: 4910

✓ Checkpoint 13 saved
Images in checkpoint: 512
Rows: 4302

✓ Checkpoint 14 saved
Images in checkpoint: 512
Rows: 4994

✓ Checkpoint 15 saved
Images in checkpoint: 512
Rows: 4030

✓ Checkpoint 16 saved
Images in checkpoint: 512
Rows: 4452

✓ Checkpoint 17 saved
Images in checkpoint: 512


In [ ]:
# ============================================================
# CELL 11 — MERGE RT-DETR-X CHECKPOINTS
# ============================================================

print("=" * 70)
print("MERGING RT-DETR-X CHECKPOINTS")
print("=" * 70)

checkpoint_files = sorted(
    CHECKPOINT_DIR.glob(
        "checkpoint_*.csv"
    ),
    key=lambda p: int(
        re.search(
            r"(\d+)",
            p.stem
        ).group()
    )
)

print(
    "\nCheckpoint files found:",
    len(checkpoint_files)
)

assert len(checkpoint_files) > 0, (
    "No RT-DETR checkpoints found."
)

checkpoint_dfs = []

for file in checkpoint_files:

    df = pd.read_csv(
        file
    )

    print(
        f"\n{file.name}"
    )

    print(
        "Rows:",
        len(df)
    )

    print(
        "Unique images:",
        df[
            "image_id"
        ].nunique()
    )

    checkpoint_dfs.append(
        df
    )

# ------------------------------------------------------------
# MERGE
# ------------------------------------------------------------

merged_df = pd.concat(
    checkpoint_dfs,
    ignore_index=True
)

print("\n" + "=" * 70)
print("MERGING DATA")
print("=" * 70)

print(
    "\nTotal records before cleanup:",
    len(merged_df)
)

# ------------------------------------------------------------
# REMOVE EXACT DUPLICATES
# ------------------------------------------------------------

before = len(
    merged_df
)

merged_df = (
    merged_df
    .drop_duplicates()
    .reset_index(drop=True)
)

duplicates_removed = (
    before -
    len(merged_df)
)

print(
    "\nDuplicate rows removed:",
    duplicates_removed
)

# ------------------------------------------------------------
# SORT
# ------------------------------------------------------------

merged_df = (
    merged_df
    .sort_values(
        by=[
            "image_id",
            "confidence"
        ],
        ascending=[
            True,
            False
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

merged_df.to_csv(
    FINAL_OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("FINAL FILE SAVED")
print("=" * 70)

print(
    "\nPath:"
)

print(
    FINAL_OUTPUT_FILE
)

print(
    "\nFinal shape:"
)

print(
    merged_df.shape
)

print(
    "\nColumns:"
)

print(
    merged_df.columns.tolist()
)

MERGING RT-DETR-X CHECKPOINTS

Checkpoint files found: 78

checkpoint_00001.csv
Rows: 4059
Unique images: 512

checkpoint_00002.csv
Rows: 3634
Unique images: 512

checkpoint_00003.csv
Rows: 3686
Unique images: 512

checkpoint_00004.csv
Rows: 4496
Unique images: 512

checkpoint_00005.csv
Rows: 3776
Unique images: 512

checkpoint_00006.csv
Rows: 4115
Unique images: 512

checkpoint_00007.csv
Rows: 4286
Unique images: 512

checkpoint_00008.csv
Rows: 5189
Unique images: 512

checkpoint_00009.csv
Rows: 4959
Unique images: 512

checkpoint_00010.csv
Rows: 5109
Unique images: 512

checkpoint_00011.csv
Rows: 4753
Unique images: 512

checkpoint_00012.csv
Rows: 4910
Unique images: 512

checkpoint_00013.csv
Rows: 4302
Unique images: 512

checkpoint_00014.csv
Rows: 4994
Unique images: 512

checkpoint_00015.csv
Rows: 4030
Unique images: 512

checkpoint_00016.csv
Rows: 4452
Unique images: 512

checkpoint_00017.csv
Rows: 6380
Unique images: 512

checkpoint_00018.csv
Rows: 6836
Unique images: 512

check

In [ ]:
# ============================================================
# CELL 12 — RT-DETR-X FINAL VERIFICATION
# ============================================================

print("=" * 70)
print("RT-DETR-X FINAL VERIFICATION")
print("=" * 70)

assert FINAL_OUTPUT_FILE.exists(), (
    "Final RT-DETR-X output file not found."
)

final_df = pd.read_csv(
    FINAL_OUTPUT_FILE
)

# ------------------------------------------------------------
# IMAGE COUNTS
# ------------------------------------------------------------

total_unique_images = (
    final_df[
        "image_id"
    ].nunique()
)

images_with_detections = (
    final_df[
        final_df[
            "object_name"
        ].notna()
    ][
        "image_id"
    ]
    .nunique()
)

zero_detection_images = (
    final_df[
        final_df[
            "object_name"
        ].isna()
    ][
        "image_id"
    ]
    .nunique()
)

# ------------------------------------------------------------
# DETECTION ROWS
# ------------------------------------------------------------

detection_records = (
    final_df[
        "object_name"
    ]
    .notna()
    .sum()
)

# ------------------------------------------------------------
# COVERAGE
# ------------------------------------------------------------

coverage = (
    images_with_detections
    / total_unique_images
    * 100
)

zero_rate = (
    zero_detection_images
    / total_unique_images
    * 100
)

# ------------------------------------------------------------
# CONFIDENCE
# ------------------------------------------------------------

confidence_values = (
    final_df[
        "confidence"
    ]
    .dropna()
)

average_confidence = (
    confidence_values.mean()
)

median_confidence = (
    confidence_values.median()
)

# ------------------------------------------------------------
# OBJECT CLASSES
# ------------------------------------------------------------

unique_classes = (
    final_df[
        "object_name"
    ]
    .dropna()
    .nunique()
)

# ------------------------------------------------------------
# AVG OBJECTS PER IMAGE
# ------------------------------------------------------------

average_objects_per_image = (
    detection_records
    / total_unique_images
)

# ------------------------------------------------------------
# VERIFY EXPECTED IMAGE COUNT
# ------------------------------------------------------------

print(
    "\nFinal detection file exists:",
    FINAL_OUTPUT_FILE.exists()
)

print(
    "\nFinal shape:",
    final_df.shape
)

print(
    "\nColumns:"
)

print(
    final_df.columns.tolist()
)

print(
    "\nUnique images:"
)

print(
    total_unique_images
)

print(
    "\nImages with detections:"
)

print(
    images_with_detections
)

print(
    "\nZero detection images:"
)

print(
    zero_detection_images
)

print(
    "\nDetection coverage:"
)

print(
    f"{coverage:.4f}%"
)

print(
    "\nZero detection rate:"
)

print(
    f"{zero_rate:.4f}%"
)

print(
    "\nTotal detection records:"
)

print(
    detection_records
)

print(
    "\nAverage detections per image:"
)

print(
    f"{average_objects_per_image:.4f}"
)

print(
    "\nAverage confidence:"
)

print(
    f"{average_confidence:.4f}"
)

print(
    "\nMedian confidence:"
)

print(
    f"{median_confidence:.4f}"
)

print(
    "\nUnique object classes:"
)

print(
    unique_classes
)

# ------------------------------------------------------------
# FINAL ASSERTIONS
# ------------------------------------------------------------

assert total_unique_images == 39874, (
    f"Expected 39,874 unique images, "
    f"found {total_unique_images}"
)

assert (
    images_with_detections
    +
    zero_detection_images
    ==
    39874
), (
    "Detection and zero-detection image "
    "counts do not match total images."
)

assert detection_records > 0, (
    "No detection records found."
)

print("\n" + "=" * 70)
print("✓ MODEL 3 — RT-DETR-X COMPLETE")
print("=" * 70)

print(
    "\nFinal file:"
)

print(
    FINAL_OUTPUT_FILE
)

RT-DETR-X FINAL VERIFICATION

Final detection file exists: True

Final shape: (496803, 9)

Columns:
['image_id', 'image_path', 'object_name', 'confidence', 'xmin', 'ymin', 'xmax', 'ymax', 'objects_detected']

Unique images:
39874

Images with detections:
39868

Zero detection images:
6

Detection coverage:
99.9850%

Zero detection rate:
0.0150%

Total detection records:
496797

Average detections per image:
12.4592

Average confidence:
0.5215

Median confidence:
0.4231

Unique object classes:
80

✓ MODEL 3 — RT-DETR-X COMPLETE

Final file:
/content/drive/MyDrive/Image_captioning/Outputs/RTDETR_Detection/Final/rtdetr_x_detections_full.csv
